# Tahap 4: Data Reduction & Feature Engineering
**Silabus:** Jiawei Han - Data Mining: Concepts and Techniques, Ch. 3 (Data Preprocessing - Part 2)  
**DataFrame Target:** `df_clean` (Data Hasil Preprocessing Step 2 & 3)  
**Fokus:**  
1. **Feature Extraction**: Pembuatan fitur komposit (`total_stay_nights`, `is_room_changed`) & Pembedaan Teoretis Feature Selection vs Feature Extraction.  
2. **Principal Component Analysis (PCA)**: Standardisasi Z-score, Matriks Kovariansi, Eigendecomposition, Tabel Explained Variance, Scree Plot, dan Proyeksi Subruang 2D (PC1 vs PC2) berdasar `is_canceled`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from IPython.display import display, Markdown

# 0. Memuat & Menyiapkan df_clean (Sesuai Step 2 & 3 Preprocessing)
csv_file = "hotel_booking_data.csv"
df_raw = pd.read_csv(csv_file)

cols_to_drop = ['name', 'email', 'phone-number', 'credit_card', 'reservation_status', 'reservation_status_date']
df_clean = df_raw.drop(columns=[c for c in cols_to_drop if c in df_raw.columns]).drop_duplicates()

adr_anom = (df_clean['adr'] < 0) | (df_clean['adr'] > 1000)
total_guests = df_clean['adults'] + df_clean['children'].fillna(0) + df_clean['babies'].fillna(0)
df_clean = df_clean[~adr_anom & (total_guests > 0)].copy()

df_clean['children'] = df_clean['children'].fillna(0).astype(int)
df_clean['country'] = df_clean['country'].fillna('Unknown')
df_clean['agent'] = df_clean['agent'].fillna(0)
df_clean['company'] = df_clean['company'].fillna(0)

# =========================================================
# 1. FEATURE EXTRACTION (SLIDE 21-23)
# =========================================================
print("=" * 80)
print("1. FEATURE EXTRACTION (PEMBUATAN FITUR KOMPOSIT BARU)")
print("=" * 80)

# Fitur 1: Total malam menginap
df_clean['total_stay_nights'] = df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']

# Fitur 2: Indikator perubahan tipe kamar (Room Mismatch)
df_clean['is_room_changed'] = (df_clean['reserved_room_type'] != df_clean['assigned_room_type']).astype(int)

print(f"▶ Fitur 'total_stay_nights' berhasil dibuat (min: {df_clean['total_stay_nights'].min()}, max: {df_clean['total_stay_nights'].max()})")
print(f"▶ Fitur 'is_room_changed' berhasil dibuat (Jumlah kamar diubah/upgrade: {df_clean['is_room_changed'].sum():,} baris)")


# =========================================================
# 2. PRINCIPAL COMPONENT ANALYSIS (PCA) STEP-BY-STEP (SLIDE 15-20)
# =========================================================
print("\n" + "=" * 80)
print("2. PRINCIPAL COMPONENT ANALYSIS (PCA)")
print("=" * 80)

pca_cols = [
    'lead_time', 'total_stay_nights', 'adults', 'children',
    'booking_changes', 'adr', 'days_in_waiting_list', 'total_of_special_requests'
]

# Langkah 1: Z-score Standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[pca_cols])
print("▶ Langkah 1 (Standardisasi): Z-score dihitung (Mean = 0, Std = 1 untuk seluruh 8 atribut numerik).")

# Langkah 2 & 3: Matriks Kovariansi & Eigendecomposition via Sklearn PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

eigenvalues = pca.explained_variance_
evr = pca.explained_variance_ratio_
cum_evr = np.cumsum(evr)

# Langkah 4: Tabel Explained Variance Ratio & Scree Plot
pca_summary = pd.DataFrame({
    'Komponen Utama': [f'PC{i+1}' for i in range(len(pca_cols))],
    'Eigenvalue (λ)': eigenvalues.round(4),
    'Explained Variance Ratio (%)': (evr * 100).round(2),
    'Cumulative Variance (%)': (cum_evr * 100).round(2)
})

print("\n--- Tabel Ringkasan Varians Komponen Utama (PCA Summary) ---")
display(pca_summary)

# Scree Plot / Cumulative Variance Plot
fig, ax1 = plt.subplots(figsize=(9, 5))

# Barplot individual EVR
ax1.bar(pca_summary['Komponen Utama'], pca_summary['Explained Variance Ratio (%)'], color='skyblue', alpha=0.8, label='Individual EVR (%)')
ax1.set_xlabel('Komponen Utama (Principal Components)', fontweight='bold')
ax1.set_ylabel('Individual Explained Variance Ratio (%)', color='blue', fontweight='bold')
ax1.tick_params(axis='y', labelcolor='blue')

# Lineplot cumulative EVR
ax2 = ax1.twinx()
ax2.plot(pca_summary['Komponen Utama'], pca_summary['Cumulative Variance (%)'], color='red', marker='o', linewidth=2, label='Cumulative Variance (%)')
ax2.set_ylabel('Cumulative Variance (%)', color='red', fontweight='bold')
ax2.tick_params(axis='y', labelcolor='red')
ax2.set_ylim(0, 105)

plt.title("Scree Plot & Cumulative Explained Variance Ratio", fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

# Langkah 5: Proyeksi Subruang 2D (PC1 vs PC2)
df_pca_2d = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'is_canceled': df_clean['is_canceled'].values
})

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_pca_2d, x='PC1', y='PC2', hue='is_canceled',
    palette={0: '#2ecc71', 1: '#e74c3c'}, alpha=0.5, s=20
)
plt.title("Proyeksi Subruang 2D PCA (PC1 vs PC2) berdasar Status Pembatalan", fontsize=13, fontweight='bold')
plt.xlabel(f"PC1 ({evr[0]*100:.2f}% Varians)", fontweight='bold')
plt.ylabel(f"PC2 ({evr[1]*100:.2f}% Varians)", fontweight='bold')
plt.legend(title='Status Pembatalan', labels=['Not Canceled (0)', 'Canceled (1)'])
plt.tight_layout()
plt.show()

### Evaluasi Semantik & Teoretis Tahap 4 (Data Reduction & Feature Engineering)

#### 1. Pembedaan Teoretis: Feature Selection vs. Feature Extraction (Konsep Jiawei Han Slide 21-23)
- **Feature Selection**:
  - Memilih subset dari fitur asli *tanpa mengubah bentuk representasi fisik* atribut tersebut (contoh: dari 10 fitur, diambil 5 fitur terbaik dan 5 sisanya dibuang).
- **Feature Extraction**:
  - Mentransformasikan atau menggabungkan ruang fitur asal menjadi set fitur baru yang berdimensi lebih rendah atau lebih kaya makna semantik.
  - **Kasus pada Project Ini**:
    1. `total_stay_nights`: Memadatkan relasi dua dimensi (`stays_in_weekend_nights` + `stays_in_week_nights`) menjadi representasi tunggal yang secara langsung mengukur total durasi menginap pelanggan.
    2. `is_room_changed`: Mengabstraksikan perbandingan string kategorikal (`reserved_room_type` vs `assigned_room_type`) menjadi fitur biner tunggal (0 = Kamar Sesuai, 1 = Kamar Diubah/Upgrade).
    3. `PCA`: Mentransformasikan 8 atribut numerik asli yang saling berkorelasi menjadi komponen ortogonal linier baru (PC1 s.d. PC8) tanpa kehilangan variansi informasi secara drastis.

---

#### 2. Interpretasi Hasil PCA & Proyeksi Subruang 2D (Slide 15-20)
- **Karakteristik Komponen Utama**:
  - **PC1** mampu menjelaskan **20.07%** dari total variansi data dengan Eigenvalue $\lambda_1 = 1.6059$.
  - **PC2** mampu menjelaskan **16.42%** dari total variansi data dengan Eigenvalue $\lambda_2 = 1.3138$.
  - Dua komponen utama pertama (PC1 & PC2) secara akumulatif menangkap **36.50%** dari total variansi data 8 dimensi.
  - Untuk mempertahankan variansi data $> 70\%$ (misal untuk reduksi dimensi sebelum klasifikasi ML), dibutuhkan **5 Komponen Utama (PC1 s.d. PC5)** yang merangkum **74.06%** akumulasi variansi.

- **Interpretasi Visual Scatter Plot 2D (PC1 vs PC2)**:
  - Distribusi titik data pada ruang 2D memperlihatkan overlap/tumpang tindih yang signifikan antara reservasi yang dibatalkan (`is_canceled = 1`, merah) dan tidak dibatalkan (`is_canceled = 0`, hijau).
  - Hal ini menunjukkan bahwa **reduksi linier murni ke 2 dimensi (PCA 2D) belum cukup kuat untuk memisahkan target secara sempurna**, sehingga pemodelan machine learning nantinya membutuhkan fitur kategorikal lain (`market_segment`, `deposit_type`, `is_room_changed`) atau model non-linear (seperti Tree-based ensemble: Random Forest/XGBoost).